### RAG System from Scratch with Langchain and Python

In [ ]:
# Chunking


import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Loading the document
with open("../data/raw/rag_notebook.txt") as f:
    knowledge_text = f.read()

# Initializing the Text Splitter
# This splitter is smart. It tries to split on paragraphs ("\n\n"), then newlines ("\n"), then spaces (" "), to keep semantically related text together as much as possible.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,  # Max size of a chunk
    chunk_overlap=20, # Overlap to maintain context between chunks
    length_function=len
)

# Creating the chunks
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ---\n{chunk}\n")

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We have 21 chunks:
--- Chunk 1 ---
- Consider these Moon Facts…

--- Chunk 2 ---
It takes 27 1/3 days for the Moon to go around Earth one time; called the Sidereal Month. From new Moon to the next new Moon takes 29 1/2 days;

--- Chunk 3 ---
takes 29 1/2 days; called the Synodic Month. Why this amount of time? Earth has also moved through space and the Moon has to catch up with that

--- Chunk 4 ---
catch up with that starting position. Think of our word month – from the term Moonth.

--- Chunk 5 ---
As the Moon orbits Earth, roughly the same side of the Moon faces Earth. This means that one (1) lunar rotation equals one (1) lunar revolution. Like

--- Chunk 6 ---
revolution. Like Earth, one half (1/2 or 50%) of the Moon is always illuminated by the Sun

--- Chunk 7 ---
- Phases of the Moon

--- Chunk 8 ---
We see the Moon go through its phases due to the position of the Earth, Moon, and Sun relative to each other, NOT due to Clouds, the Moon moving

--- Chunk 9 ---
the Moon moving clo

In [4]:
# Embeddings - turning these text chunks into numbers (vectors)
from sentence_transformers import SentenceTransformer

# 'all-MiniLM-L6-v2' is a good, small embedding model.
# It runs 100% on your local machine.
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embedding all chunks
# This will take a moment as it "reads" and "understands" each chunk.
chunk_embeddings = model.encode(chunks)

print(f"Shape of the embeddings: {chunk_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20327.17it/s]


Shape of the embeddings: (21, 384)


In [8]:
chunk_embeddings[0].shape

(384,)

In [9]:
# Vector Store with FAISS
# Now after having created the embeddings vector we need a database to store them in a way we can search by similarity = FAISS

import faiss
import numpy as np

# dimension of our vectors are 384 (the size of the embedding model output)
d = chunk_embeddings.shape[1]

# Creating the FAISS index
# IndexFlatL2 is the simplest, most basic index - It calculates the exact distance (L2 distance) between our query and all vectors.
index = faiss.IndexFlatL2(d)

# Adding our chunk embeddings to the index. We must convert to float32 for FAISS
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 21 vectors.


In [14]:
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

load_dotenv()  # loads from .env file in the project root

hf_token = os.getenv("HF_TOKEN")
headers = {"Authorization": f"Bearer {hf_token}"}

client = InferenceClient(token=hf_token)

In [15]:
# Retrieve, Augment, Generate

def answer_question(query):
    # RETRIEVING

    # Embedding the user's query
    query_embedding = model.encode([query]).astype('float32')
    k = 2 # # Searching the FAISS index for the top k (e.g., k=2) most similar chunks
    distances, indices = index.search(query_embedding, k)

    # Getting the actual text chunks from the original 'chunks' list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # AUGMENTING + GENERATING
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"""Answer the following question using *only* the provided context. If the answer is not in the context, say "I don't have that information."

            Context:
            {context}

            Question:
            {query}

            Answer:
            """
        }],
        model="meta-llama/Meta-Llama-3-8B-Instruct", # 'google/flan-t5-small'
        max_tokens=150
    )

    print(f"--- CONTEXT ---\n{context}\n")
    return result.choices[0].message.content.strip()

In [ ]:
# Asking the Question
query_1 = "Can the moon be seen in the daytime?" # Answer is yes and is present in the data
print(f"Query: {query_1}")
print(f"Answer: {answer_question(query_1)}\n")

Query: Can the moon be seen in the daytime?
--- CONTEXT ---
- More about the Moon…

Can we see the Moon in the daytime? Yes!

Are there times when we cannot see the Moon at all? Yes, at New Moon and sometimes right before and right after the New Moon because the thin

Answer: Yes!



In [17]:
# Asking the Question that is not present in the data
query_2 = "What is the distance from Earth to the Sun?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")

Query: What is the distance from Earth to the Sun?
--- CONTEXT ---
miles. Perigee is when the Moon is closest to the Earth and Apogee is when the Moon is farthest from Earth.

Does the Moon keep a constant distance from Earth? No, it varies from a Perigee of about 225,000 miles to an Apogee of about 243,000 miles. Perigee

Answer: I don't have that information.

